In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

In [ ]:
from pipeline.pipeline_orchestrator import PipelineOrchestrator
from pipeline.dataclasses.pipeline_config import PipelineConfig, BranchConfig, ModelConfig, TrainerConfig, TrainValTestSplitRatio, AugmentationConfig, CustomSplitConfig
from pipeline.dataclasses.enums import ModelType, FilterType, DatasetLoaderType, AugmentationType, SplitType
from pipeline.dataclasses.training_configs import AdamConfig, ReduceLROnPlateauConfig

In [ ]:
pipeline_config = PipelineConfig(
    pipeline_id="one_epoch_full_branch_expirement",
    data_path=Path(proj_dir / "data"),
    split_config=CustomSplitConfig(
        train_csv_path=Path(proj_dir / "data" / "train_df.csv"),
        val_csv_path=Path(proj_dir / "data" / "val_df.csv"),
        test_csv_path=Path(proj_dir / "data" / "test_df.csv"),
    ),
    seed=42,
    branches=[
        # ──────────────────────────────────────────
        # RGB BRANCHES
        # ──────────────────────────────────────────
        BranchConfig(
            branch_id="rgb_resnet50_wrs",
            sampler_weights=(0.7, 0.3), 
            branch_model_config=ModelConfig(
                model=ModelType.RESNET_50,
                filter_type=FilterType.RGB,
                num_classes=2,
                pretrained=True,
                freeze_weights=False,
            ),
            trainer_config=TrainerConfig(
                batch_size=16,
                num_epochs=45,
                learning_rate=0.001,
                criterion_weights=(1/0.8, 1/0.2),
                optimizer=AdamConfig(),
                scheduler=ReduceLROnPlateauConfig(),
            ),
            dataset_loader=DatasetLoaderType.RGB,
            augmentation=AugmentationConfig(
                train=[
                    AugmentationType.RESIZE,
                    AugmentationType.CENTER_CROP,
                    AugmentationType.RANDOM_HORIZONTAL_FLIP,
                    AugmentationType.RANDOM_VERTICAL_FLIP,
                    AugmentationType.RANDOM_RESIZE_CROP,
                    AugmentationType.COLOR_JITTER,
                ],
                val=[],
            ),
        ),
        BranchConfig(
            branch_id="gabor_resnet50_wrs",
            sampler_weights=(0.7, 0.3), 
            branch_model_config=ModelConfig(
                model=ModelType.RESNET_50,
                filter_type=FilterType.GABOR,
                num_classes=2,
                pretrained=True,
                freeze_weights=False,
            ),
            trainer_config=TrainerConfig(
                batch_size=16,
                num_epochs=45,
                learning_rate=0.001,
                criterion_weights=(1/0.8, 1/0.2),
                optimizer=AdamConfig(),
                scheduler=ReduceLROnPlateauConfig(),
            ),
            dataset_loader=DatasetLoaderType.FILTER,
            augmentation=AugmentationConfig(
                train=[
                    AugmentationType.RESIZE,
                ],
                val=[],
            ),
        ),
        # BranchConfig(
        #     branch_id="rgb_resnet50_finetune_wrs",
        #     branch_model_config=ModelConfig(
        #         model=ModelType.RESNET_50,
        #         filter_type=FilterType.RGB,
        #         num_classes=2,
        #         pretrained=True,
        #         freeze_weights=False,
        #     ),
        #     trainer_config=TrainerConfig(
        #         batch_size=16,
        #         num_epochs=20,
        #         learning_rate=0.0001,
        #         optimizer=AdamConfig(),
        #         scheduler=ReduceLROnPlateauConfig(),
        #     ),
        #     dataset_loader=DatasetLoaderType.RGB,
        #     augmentation=AugmentationConfig(
        #         train=[
        #             AugmentationType.RESIZE,
        #             AugmentationType.CENTER_CROP,
        #             AugmentationType.RANDOM_HORIZONTAL_FLIP,
        #             AugmentationType.RANDOM_VERTICAL_FLIP,
        #             AugmentationType.RANDOM_RESIZE_CROP,
        #         ],
        #         val=[],
        #     ),
        # ),
        # ──────────────────────────────────────────
        # GABOR BRANCHES
        # ──────────────────────────────────────────
        # BranchConfig(
        #     branch_id="gabor_resnet50",
        #     branch_model_config=ModelConfig(
        #         model=ModelType.RESNET_50,
        #         filter_type=FilterType.GABOR,
        #         num_classes=2,
        #         pretrained=False,
        #         freeze_weights=False,
        #     ),
        #     trainer_config=TrainerConfig(
        #         batch_size=12,
        #         num_epochs=40,
        #         learning_rate=0.001,
        #         optimizer=AdamConfig(),
        #         scheduler=ReduceLROnPlateauConfig(),
        #     ),
        #     dataset_loader=DatasetLoaderType.FILTER,
        # ),
        # ──────────────────────────────────────────
        # GLCM BRANCHES
        # ──────────────────────────────────────────
        # BranchConfig(
        #     branch_id="glcm_resnet50",
        #     branch_model_config=ModelConfig(
        #         model=ModelType.RESNET_50,
        #         filter_type=FilterType.GLCM,
        #         num_classes=2,
        #         pretrained=False,
        #         freeze_weights=False,
        #     ),
        #     trainer_config=TrainerConfig(
        #         batch_size=16,
        #         num_epochs=40,
        #         learning_rate=0.001,
        #         optimizer=AdamConfig(),
        #         scheduler=ReduceLROnPlateauConfig(),
        #     ),
        #     dataset_loader=DatasetLoaderType.FILTER,
        # ),
        # ──────────────────────────────────────────
        # LBP BRANCHES
        # ──────────────────────────────────────────
        # BranchConfig(
        #     branch_id="lbp_resnet50",
        #     branch_model_config=ModelConfig(
        #         model=ModelType.RESNET_50,
        #         filter_type=FilterType.LBP,
        #         num_classes=2,
        #         pretrained=False,
        #         freeze_weights=False,
        #     ),
        #     trainer_config=TrainerConfig(
        #         batch_size=16,
        #         num_epochs=30,
        #         learning_rate=0.001,
        #         optimizer=AdamConfig(),
        #         scheduler=ReduceLROnPlateauConfig(),
        #     ),
        #     dataset_loader=DatasetLoaderType.FILTER,
        # ),
    ],
)

In [ ]:
pipeline_runner = PipelineOrchestrator(
    pipeline_config=pipeline_config, 
    project_root_path=proj_dir
    )

In [ ]:
run = pipeline_runner.run()

In [ ]:
run